In [ ]:
import pandas as pd
round_df = round_df.drop_duplicates().drop_duplicates(subset='match_id', keep='first')


print("round:", round_df.shape)
print("\n cols:", list(round_df.columns))
print("\n--- top five  ---")
print(round_df.head())
print("\n--- uniqe name ---")
print(sorted(round_df['name'].dropna().unique()))

شکل round: (9243, 5)

ستون‌ها: ['match_id', 'round_id', 'name', 'slug', 'cup_round_type']

--- ۵ ردیف اول ---
   match_id  round_id         name         slug  cup_round_type
0  11998445         5  Round of 16  round-of-16             8.0
1  11998446         5  Round of 16  round-of-16             8.0
2  11998447         5  Round of 16  round-of-16             8.0
3  11998448         5  Round of 16  round-of-16             8.0
4  11998449         5  Round of 16  round-of-16             8.0

--- مقادیر یکتای name ---
['Final', 'Qualification round 1', 'Qualification round 2', 'Quarterfinal', 'Quarterfinals', 'Round of 128', 'Round of 16', 'Round of 32', 'Round of 64', 'Semifinal', 'Semifinals']


In [ ]:
finals_df = round_df[round_df['name'] == 'Final'].copy()

print("rows with name=Final:", len(finals_df))
print("uniqe match_id count:", finals_df['match_id'].nunique())
print("duplicated match_id :", finals_df['match_id'].duplicated().sum())

تعداد ردیف با name=Final: 260
تعداد match_id یکتا: 260
تعداد match_id تکراری: 0


In [ ]:
event_df= pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\event.csv")
home_df = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\home_team.csv")
away_df = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\away_team.csv")

event_df = event_df.drop_duplicates().drop_duplicates(subset='match_id', keep='first')
home_df = home_df.drop_duplicates().drop_duplicates(subset='match_id', keep='first')
away_df = away_df.drop_duplicates().drop_duplicates(subset='match_id', keep='first')

event_clean = event_df[['match_id', 'winner_code', 'start_datetime']].dropna(subset=['winner_code'])
home_clean = home_df[['match_id', 'player_id', 'full_name']]
away_clean = away_df[['match_id', 'player_id', 'full_name']]

#keep final matches only
finals_merged = finals_df[['match_id']].merge(event_clean, on='match_id', how='inner')
finals_merged = finals_merged.merge(home_clean, on='match_id', how='inner')
finals_merged = finals_merged.merge(away_clean, on='match_id', how='inner', suffixes=('_home', '_away'))

print("تعداد فینال‌هایی که همه اطلاعاتشون کامله:", len(finals_merged))
print(finals_merged.head())

تعداد فینال‌هایی که همه اطلاعاتشون کامله: 225
   match_id  winner_code  start_datetime  player_id_home      full_name_home  \
0  12002073          1.0      1707051600           64496   Ostapenko, Jelena   
1  12002069          1.0      1707042600          318575     Shnaider, Diana   
2  11998663          1.0      1707012000           79467        Jasika, Omar   
3  11999016          1.0      1707051600          200159     Rodionov, Jurij   
4  12038725          2.0      1707031800          329533  Vilicich, Bautista   

   player_id_away           full_name_away  
0           67759   Alexandrova, Ekaterina  
1           45584                 Zhu, Lin  
2           58369               Bolt, Alex  
3          235576       Nakashima, Brandon  
4          177686  Barranco Cosano, Javier  


In [ ]:
import numpy as np

# choose winner 
finals_merged['winner_id'] = np.where(finals_merged['winner_code'] == 1, finals_merged['player_id_home'], finals_merged['player_id_away'])
finals_merged['winner_name'] = np.where(finals_merged['winner_code'] == 1, finals_merged['full_name_home'], finals_merged['full_name_away'])

# تبدیل start_datetime (که یه unix timestamp هست، یعنی ثانیه از 1970) به تاریخ واقعی
finals_merged['match_date'] = pd.to_datetime(finals_merged['start_datetime'], unit='s')
finals_merged['year_month'] = finals_merged['match_date'].dt.to_period('M')

print(finals_merged[['match_id', 'winner_name', 'match_date', 'year_month']].head(10))

   match_id              winner_name          match_date year_month
0  12002073        Ostapenko, Jelena 2024-02-04 13:00:00    2024-02
1  12002069          Shnaider, Diana 2024-02-04 10:30:00    2024-02
2  11998663             Jasika, Omar 2024-02-04 02:00:00    2024-02
3  11999016          Rodionov, Jurij 2024-02-04 13:00:00    2024-02
4  12038725  Barranco Cosano, Javier 2024-02-04 07:30:00    2024-02
5  12038651       Moroni, Gian Marco 2024-02-04 08:30:00    2024-02
6  12039714         Tiurnev, Evgenii 2024-02-04 08:30:00    2024-02
7  12038552            Gengel, Marek 2024-02-04 08:00:00    2024-02
8  12039364         Bouquier, Arthur 2024-02-04 09:55:00    2024-02
9  12039718    Miyazaki, Yuriko Lily 2024-02-04 13:30:00    2024-02


In [9]:
champion_month_counts = finals_merged.groupby(['winner_id', 'winner_name', 'year_month']).size().reset_index(name='titles')
champion_month_counts = champion_month_counts.sort_values('titles', ascending=False)

print(champion_month_counts.head(10))

     winner_id                        winner_name year_month  titles
14       50901                      Popko, Dmitry    2024-02       3
180     372311                       Nicod, Jakub    2024-03       3
124     230049              Jianu, Filip Cristian    2024-03       3
99      202572                      Gengel, Marek    2024-02       3
17       52533                  Jakupovic, Dalila    2024-03       2
40       82133  Dellien Velasco, Murkel Alejandro    2024-02       2
26       63642              Kwiatkowski, Thai-Son    2024-03       2
71      152766                      Helgo, Malene    2024-03       2
100     205282                  Kužmová, Katarína    2024-02       2
154     293708                      Boisson, Loïs    2024-03       2


**QUESTION TEN**

In [14]:
import pandas as pd

home_df = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\home_team.csv")
away_df = pd.read_csv(r"C:\Users\Personal Camputer\OneDrive\Desktop\tennis\away_team.csv")

players = pd.concat([
    home_df[['player_id', 'full_name', 'height', 'current_rank']],
    away_df[['player_id', 'full_name', 'height', 'current_rank']]
], ignore_index=True).drop_duplicates(subset='player_id')

print("تعداد کل بازیکن یکتا:", len(players))
print("\nتعداد null در height:", players['height'].isnull().sum())
print("تعداد null در current_rank:", players['current_rank'].isnull().sum())

print("\n--- آمار توصیفی current_rank ---")
print(players['current_rank'].describe())

تعداد کل بازیکن یکتا: 2644

تعداد null در height: 1327
تعداد null در current_rank: 71

--- آمار توصیفی current_rank ---
count    2573.000000
mean      745.097940
std       454.232715
min         1.000000
25%       343.000000
50%       717.000000
75%      1156.000000
max      1858.000000
Name: current_rank, dtype: float64


In [15]:
players_valid = players.dropna(subset=['height', 'current_rank'])

print("تعداد بازیکنانی که هم height هم rank دارن:", len(players_valid))

correlation = players_valid['height'].corr(players_valid['current_rank'])
print(f"\nضریب همبستگی پیرسون بین height و current_rank: {correlation:.4f}")

تعداد بازیکنانی که هم height هم rank دارن: 1307

ضریب همبستگی پیرسون بین height و current_rank: 0.1051


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

plt.figure(figsize=(10, 6))

sns.regplot(
    data=players_valid,
    x='height',
    y='current_rank',
    scatter_kws={'alpha': 0.4, 's': 30},   # alpha کم چون نقاط زیادن و روی هم می‌افتن
    line_kws={'color': 'red'}
)

plt.title(f"Height vs Current Rank (n={len(players_valid)}, r={correlation:.3f})", fontsize=13)
plt.xlabel("Height (m)")
plt.ylabel("Current Rank (lower is better)")

plt.gca().invert_yaxis()  # چون rank=1 بهترینه، برعکس کردن محور y باعث میشه "بهتر" یعنی بالاتر تو نمودار

plt.tight_layout()
plt.show()